# ResNet101 — Deep Residual Learning (TensorFlow / Keras)
**Paper:** Deep Residual Learning for Image Recognition (He et al., CVPR 2016)

| Property | Value |
|----------|-------|
| Framework | TensorFlow / Keras |
| Block type | Bottleneck (post-activation, expansion=4) |
| Stage depths | 3 - 4 - **23** - 3 |
| Final feature dim | 2048 |
| Parameters | ~44.5M |
| Top-1 (ImageNet) | ~76.4% |
| Input size | 224×224 |
| vs ResNet50 | Stage 3: 23 blocks instead of 6 |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import roc_curve, auc, roc_auc_score
from sklearn.preprocessing import label_binarize
print(f'TensorFlow version : {tf.__version__}')
print(f'GPU available      : {bool(tf.config.list_physical_devices("GPU"))}')

In [ ]:
# Cell 2 — ResNet101 Architecture (from scratch)
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers


def _bottleneck(x, filters, stride=1, name=None):
    """
    ResNet Bottleneck block (post-activation): Conv1x1 -> Conv3x3 -> Conv1x1 + residual.
    Projection shortcut applied when stride != 1 or channels mismatch.
    expansion = 4: output channels = filters * 4.
    """
    shortcut    = x
    out_channels = filters * 4

    if stride != 1 or x.shape[-1] != out_channels:
        shortcut = layers.Conv2D(
            out_channels, 1, strides=stride, use_bias=False,
            name=f'{name}_0_conv' if name else None)(x)
        shortcut = layers.BatchNormalization(
            name=f'{name}_0_bn' if name else None)(shortcut)

    x = layers.Conv2D(filters, 1, use_bias=False,
                      name=f'{name}_1_conv' if name else None)(x)
    x = layers.BatchNormalization(name=f'{name}_1_bn' if name else None)(x)
    x = layers.ReLU()(x)

    x = layers.Conv2D(filters, 3, strides=stride, padding='same', use_bias=False,
                      name=f'{name}_2_conv' if name else None)(x)
    x = layers.BatchNormalization(name=f'{name}_2_bn' if name else None)(x)
    x = layers.ReLU()(x)

    x = layers.Conv2D(out_channels, 1, use_bias=False,
                      name=f'{name}_3_conv' if name else None)(x)
    x = layers.BatchNormalization(name=f'{name}_3_bn' if name else None)(x)

    x = layers.Add()([x, shortcut])
    x = layers.ReLU()(x)
    return x


def build_resnet101(num_classes=1000, input_shape=(224, 224, 3)):
    """
    ResNet-101 — Deep Residual Learning for Image Recognition.
    Paper: He et al., CVPR 2016.

    Identical to ResNet-50 except Stage 3 has 23 bottleneck blocks (not 6).
    Stage depths: 3-4-23-3  (ResNet-50: 3-4-6-3)

      Stem    : Conv7x7/2 + BN + ReLU + MaxPool3x3/2  ->   64 x 56x56
      Stage 1 : 3  x Bottleneck(64)  stride=1          ->  256 x 56x56
      Stage 2 : 4  x Bottleneck(128) stride=2          ->  512 x 28x28
      Stage 3 : 23 x Bottleneck(256) stride=2          -> 1024 x 14x14
      Stage 4 : 3  x Bottleneck(512) stride=2          -> 2048 x  7x7
      Head    : GlobalAvgPool -> Dense(num_classes)
    """
    inputs = keras.Input(shape=input_shape)

    # ── Stem ──
    x = layers.ZeroPadding2D(3, name='conv1_pad')(inputs)
    x = layers.Conv2D(64, 7, strides=2, use_bias=False, name='conv1_conv')(x)  # 112x112
    x = layers.BatchNormalization(name='conv1_bn')(x)
    x = layers.ReLU(name='conv1_relu')(x)
    x = layers.ZeroPadding2D(1, name='pool1_pad')(x)
    x = layers.MaxPooling2D(3, strides=2, name='pool1_pool')(x)                # 56x56

    # ── Stage 1: 3 blocks, 64 filters ──
    x = _bottleneck(x, 64,  stride=1, name='conv2_block1')   # 256 ch
    x = _bottleneck(x, 64,  stride=1, name='conv2_block2')
    x = _bottleneck(x, 64,  stride=1, name='conv2_block3')

    # ── Stage 2: 4 blocks, 128 filters, stride=2 ──
    x = _bottleneck(x, 128, stride=2, name='conv3_block1')   # 512 ch, 28x28
    x = _bottleneck(x, 128, stride=1, name='conv3_block2')
    x = _bottleneck(x, 128, stride=1, name='conv3_block3')
    x = _bottleneck(x, 128, stride=1, name='conv3_block4')

    # ── Stage 3: 23 blocks, 256 filters, stride=2  (key difference vs ResNet-50) ──
    x = _bottleneck(x, 256, stride=2, name='conv4_block1')   # 1024 ch, 14x14
    for i in range(2, 24):                                    # blocks 2-23
        x = _bottleneck(x, 256, stride=1, name=f'conv4_block{i}')

    # ── Stage 4: 3 blocks, 512 filters, stride=2 ──
    x = _bottleneck(x, 512, stride=2, name='conv5_block1')   # 2048 ch, 7x7
    x = _bottleneck(x, 512, stride=1, name='conv5_block2')
    x = _bottleneck(x, 512, stride=1, name='conv5_block3')

    x = layers.GlobalAveragePooling2D(name='avg_pool')(x)
    outputs = layers.Dense(num_classes, activation='softmax', name='predictions')(x)

    return keras.Model(inputs=inputs, outputs=outputs, name='resnet101')


In [ ]:
NUM_CLASSES = 10
INPUT_SHAPE = (224, 224, 3)

model = build_resnet101(num_classes=NUM_CLASSES, input_shape=INPUT_SHAPE)
model.summary(line_length=120)

In [ ]:
dummy  = tf.random.normal((2, 224, 224, 3))
output = model(dummy, training=False)
print(f'Input  shape : {dummy.shape}')
print(f'Output shape : {output.shape}')

In [ ]:
total_params     = model.count_params()
trainable_params = int(np.sum([np.prod(v.shape) for v in model.trainable_variables]))
non_trainable    = total_params - trainable_params
print(f"{'='*48}")
print(f"  Total parameters      : {total_params:,}")
print(f"  Trainable parameters  : {trainable_params:,}")
print(f"  Non-trainable         : {non_trainable:,}")
print(f"{'='*48}")

In [ ]:
print(f"{'Layer':<55} {'Output Shape':<30} {'Params':>10}")
print('-' * 98)
total = 0
for layer in model.layers:
    params = layer.count_params()
    total += params
    try:
        out_shape = str(layer.output_shape)
    except Exception:
        out_shape = 'multiple'
    print(f"{layer.name:<55} {out_shape:<30} {params:>10,}")
print('-' * 98)
print(f"{'TOTAL':<86} {total:>10,}")

---
## Training

In [ ]:
DATA_DIR   = './data'
BATCH_SIZE = 16

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    zoom_range=0.1,
)
val_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    f'{DATA_DIR}/train',
    target_size=(224, 224),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
)
val_gen = val_datagen.flow_from_directory(
    f'{DATA_DIR}/val',
    target_size=(224, 224),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False,
)
CLASS_NAMES = list(train_gen.class_indices.keys())
print(f'Classes    : {CLASS_NAMES}')
print(f'Train size : {train_gen.samples}')
print(f'Val size   : {val_gen.samples}')

In [ ]:
EPOCHS = 30

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

callbacks = [
    keras.callbacks.ModelCheckpoint(
        'resnet101_best.keras', monitor='val_accuracy',
        save_best_only=True, verbose=1,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.1, patience=5,
        min_lr=1e-7, verbose=1,
    ),
]

history = model.fit(
    train_gen,
    epochs=EPOCHS,
    validation_data=val_gen,
    callbacks=callbacks,
    verbose=1,
)
print('Best model saved: resnet101_best.keras')

---
## Training Curves

In [ ]:
epochs_range = range(1, len(history.history['loss']) + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('ResNet101 — Training Curves', fontsize=14, fontweight='bold')

axes[0].plot(epochs_range, history.history['loss'],         'b-o', linewidth=2, markersize=4, label='Train Loss')
axes[0].plot(epochs_range, history.history['val_loss'],     'r-o', linewidth=2, markersize=4, label='Val Loss')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, history.history['accuracy'],     'b-o', linewidth=2, markersize=4, label='Train Acc')
axes[1].plot(epochs_range, history.history['val_accuracy'], 'r-o', linewidth=2, markersize=4, label='Val Acc')
axes[1].set_title('Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Inference

In [ ]:
from PIL import Image

def predict_image(image_path, top_k=5):
    img    = keras.utils.load_img(image_path, target_size=(224, 224))
    arr    = keras.utils.img_to_array(img) / 255.0
    tensor = tf.expand_dims(arr, 0)
    probs  = model(tensor, training=False)[0].numpy()
    top_idx = probs.argsort()[::-1][:top_k]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].imshow(img); axes[0].axis('off'); axes[0].set_title('Input')
    labels    = [CLASS_NAMES[i] for i in top_idx]
    top_probs = [probs[i] * 100 for i in top_idx]
    axes[1].barh(labels[::-1], top_probs[::-1])
    axes[1].set_xlabel('Confidence (%)'); axes[1].set_title(f'Top-{top_k} Predictions')
    plt.tight_layout(); plt.show()
    print(f'Predicted: {CLASS_NAMES[top_idx[0]]}  ({probs[top_idx[0]]*100:.2f}%)')

# predict_image('your_image.jpg')
print('Call: predict_image("your_image.jpg")')

---
## ROC AUC Curve

In [ ]:
val_gen.reset()
y_pred     = model.predict(val_gen, steps=len(val_gen), verbose=1)
y_true     = val_gen.classes
y_true_bin = label_binarize(y_true, classes=list(range(NUM_CLASSES)))
print(f'Predictions shape : {y_pred.shape}')
print(f'Labels shape      : {y_true_bin.shape}')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
colors  = plt.cm.tab10(np.linspace(0, 1, NUM_CLASSES))
for i in range(NUM_CLASSES):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_pred[:, i])
    ax.plot(fpr, tpr, color=colors[i], linewidth=2,
            label=f'{CLASS_NAMES[i]}  (AUC = {auc(fpr,tpr):.3f})')

macro_auc = roc_auc_score(y_true_bin, y_pred, average='macro', multi_class='ovr')
ax.plot([0,1],[0,1],'k--',linewidth=1,label='Random (AUC=0.500)')
ax.set_xlim([0,1]); ax.set_ylim([0,1.05])
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title(f'ResNet101 — ROC AUC  (Macro AUC = {macro_auc:.3f})', fontweight='bold')
ax.legend(loc='lower right', fontsize=9); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('roc_auc_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Macro AUC : {macro_auc:.4f}')